# Mosaic & merge

Combine several rasters into one:

- **`merge_rasters(src, dst)`** — mosaic many (overlapping or adjacent) rasters into a single
  raster covering their union.
- **`stack_bands(files, path=...)`** — stack several single-band rasters into one multi-band
  raster (e.g. assembling per-band Sentinel files into one image).

## Setup

In [ ]:
%matplotlib inline

import tempfile
from pathlib import Path

DATA = Path('../../../examples/data')
WORK = Path(tempfile.mkdtemp(prefix='pyramids-ops-'))
DATA.is_dir(), WORK.is_dir()

In [ ]:
from pyramids.dataset import Dataset
from pyramids.dataset.merge import merge_rasters, stack_bands

tiles = [DATA / 'crop_aligned_folder' / f'{i}.tif' for i in range(3)]
[Dataset.read_file(t).shape for t in tiles]

In [ ]:
# Each input tile is a single-band 24x29 raster on a shared grid.
# `Dataset.plot(band=0)` renders a band inline (it returns a cleopatra ArrayGlyph).
Dataset.read_file(tiles[0]).plot(band=0)

## Mosaic — `merge_rasters`

Writes one raster covering the inputs' combined extent. `method` controls how overlaps resolve (`'last'`, `'first'`, `'min'`, `'max'`, `'sum'`).

The mosaic's no-data marker is **taken from the sources**: the first value they declare wins. These tiles declare `2147483648.0`, so the mosaic declares it too. Where the sources declare nothing, a marker is chosen only if some pixel is left uncovered — there has to be something for it to mark.

In [ ]:
mosaic = WORK / 'mosaic.tif'
merge_rasters(tiles, mosaic)
Dataset.read_file(mosaic).shape

In [ ]:
# The mosaic covers the inputs' combined extent — plot it to see the merged raster.
Dataset.read_file(mosaic).plot(band=0)

## Stack bands — `stack_bands`

Combine single-band rasters into one multi-band raster (order = input order).

In [ ]:
stacked = WORK / 'stacked.tif'
stack_bands(tiles, path=stacked, band_names=['t0', 't1', 't2'])
out = Dataset.read_file(stacked)
out.band_count, out.band_names

In [ ]:
# The stacked raster carries the three tiles as bands t0/t1/t2 — plot each band.
for b in range(out.band_count):
    out.plot(band=b)

## Notes

- `no_data_value=` overrides the inherited marker, and `None` asks for none at all. Whatever it resolves to also fills the pixels no source covers, so the marker masks what it is there to mask.
- `method=` controls how overlapping pixels resolve; `'min'`, `'max'` and `'sum'` write `Float64`.
- `stack_bands` can `align=True` to snap mismatched grids before stacking.
- See also: [Reproject / resample / align](reproject-resample-align.ipynb).